# ML01 · 資料與張量基礎

機器學習的第一步：學會把資料整理成模型看得懂的形狀，並理解張量（tensor）與標準化（standardization）的基本直覺。

## 學習目標
- 能分辨特徵（feature）與標籤（label），並說出資料矩陣的形狀含義（樣本數、特徵數）。
- 能用 numpy 陣列建立、檢視與操作資料，掌握形狀（shape）與資料型別（dtype）。
- 理解特徵標準化的動機，並用 z-score 把特徵轉成可比較的尺度。
- 認識張量的概念，理解它與 numpy 陣列的關係與差異。
- 能把一批資料切成訓練集（training set）與測試集（test set），並說明為何要這樣切。

## 資料的形狀：特徵與標籤

機器學習的輸入幾乎都是一張二維表：一列（row）是一個樣本（sample），一欄（column）是一個特徵（feature）。
- 特徵：用來描述樣本的輸入數值，例如一棟建築的樓高、面積、屋齡。
- 標籤（label）：我們想預測的答案，例如這棟是不是住宅。

為什麼先學這個：建立「一列一樣本、一欄一特徵」的心智模型後，後面所有資料都能套進這個形狀。

形狀：特徵矩陣為 (樣本數 N, 特徵數 D)，標籤為長度 N 的向量。

In [ ]:
# 概念：把資料排成「N 個樣本 × D 個特徵」的二維矩陣，標籤是長度 N 的一維向量
import numpy as np

# 造一份假建築資料：每棟有 樓高(公尺) / 面積(平方公尺) / 屋齡(年) 三個特徵
features = np.array([
    [12.0, 85.0, 5.0],
    [30.0, 120.0, 12.0],
    [9.0, 60.0, 25.0],
    [45.0, 200.0, 3.0],
    [18.0, 95.0, 8.0],
])

# 標籤：1 代表住宅、0 代表非住宅（與上面每一列一一對應）
labels = np.array([1, 1, 0, 1, 0])

print("特徵矩陣（每列一棟建築）:")
print(features)
print("標籤向量:", labels)
# shape 讀作 (樣本數, 特徵數)；這裡是 5 棟、每棟 3 個特徵
print("特徵 shape:", features.shape, "  標籤 shape:", labels.shape)

## numpy 陣列複習

numpy 陣列（ndarray）是後面所有運算的底層容器。它支援索引（indexing）與切片（slicing），並能用向量化運算（vectorization）一次處理整欄，取代逐筆 for 迴圈。

為什麼重要：向量化既快又貼近數學寫法；廣播（broadcasting）讓「陣列 + 單一數字」自動套用到每個元素。

形狀：`陣列[列索引, 欄索引]`，用 `:` 代表整列或整欄。

In [ ]:
# 概念：用切片依列／欄取值，用向量化與廣播一次處理整欄
import numpy as np

features = np.array([
    [12.0, 85.0, 5.0],
    [30.0, 120.0, 12.0],
    [9.0, 60.0, 25.0],
    [45.0, 200.0, 3.0],
    [18.0, 95.0, 8.0],
])

heights = features[:, 0]      # 取第 0 欄（所有樓高）；冒號代表「所有列」
first_three = features[:3]    # 取前三棟（前三列），等同 features[0:3]

print("所有樓高:", heights)
print("前三棟:\n", first_three)

# 廣播：整欄一次加上 1（每棟樓高 +1 公尺），不需要寫迴圈
print("樓高各加 1 公尺:", heights + 1)
# 向量化：整欄面積乘以 0.9（例如打九折的可用面積）
print("面積打九折:", features[:, 1] * 0.9)

## 形狀 shape 與資料型別 dtype

shape 描述陣列每個軸（axis）的長度；`reshape` 可在不改變資料的前提下換一個形狀。資料型別（dtype）決定數值是整數還是浮點數。

為什麼重要：形狀對不上是初學者最常見的錯誤來源；整數運算會把小數截掉，標準化等計算需要浮點數。養成隨手檢查 shape 與 dtype 的習慣。

形狀：`陣列.reshape(列數, 欄數)`，其中一個維度可用 `-1` 讓 numpy 自動推算。

In [ ]:
# 概念：reshape 改形狀（資料不變），以及 int 與 float 的差別與型別轉換 cast
import numpy as np

# 造一維容積率資料（6 棟）
far = np.array([2.25, 3.0, 1.8, 4.5, 2.1, 3.6])
print("原始 shape:", far.shape)             # (6,) 是一維

col_vector = far.reshape(-1, 1)              # -1 讓 numpy 自動算列數，變成 (6, 1) 欄向量
print("reshape 後 shape:", col_vector.shape)

# 整數型樓層數
floors_int = np.array([3, 7, 5, 12, 4])
print("整數樓層 dtype:", floors_int.dtype)

# 注意：整數除法會截掉小數，例如平均每層用的概念計算會失真
print("整數相除（會截斷）:", floors_int / 2)  # numpy 除法本身會升成 float，但若用 int 累積就會掉精度

floors_float = floors_int.astype(float)      # cast 成浮點數，保留小數
print("轉成浮點 dtype:", floors_float.dtype, " 內容:", floors_float)

## 特徵標準化 standardization

不同特徵的數量級常常差很多：樓高是十幾公尺、面積是上百平方公尺。若不處理，模型容易被大數值特徵主導。

特徵標準化（standardization）用 z-score 把每個特徵轉成「平均 0、標準差 1」的尺度，讓每個特徵站在同一條起跑線上。要逐欄（依特徵）計算統計量。

公式：z = (x − 平均) / 標準差。

In [ ]:
# 概念：逐欄做 z-score 標準化，並比較標準化前後每欄的平均與分布範圍
import numpy as np

features = np.array([
    [12.0, 85.0, 5.0],
    [30.0, 120.0, 12.0],
    [9.0, 60.0, 25.0],
    [45.0, 200.0, 3.0],
    [18.0, 95.0, 8.0],
])

# axis=0 代表「沿著列方向往下壓」，得到每一欄（每個特徵）的統計量
col_mean = features.mean(axis=0)   # 每欄平均
col_std = features.std(axis=0)     # 每欄標準差

standardized = (features - col_mean) / col_std   # 廣播：每欄各自減平均、除標準差

print("標準化前 每欄平均:", np.round(col_mean, 2))
print("標準化前 每欄範圍:", np.round(features.max(axis=0) - features.min(axis=0), 2))
print("標準化後 每欄平均:", np.round(standardized.mean(axis=0), 6))   # 應接近 0
print("標準化後 每欄標準差:", np.round(standardized.std(axis=0), 6))  # 應接近 1
# 技巧：標準差可能為 0（整欄相同）會造成除以 0；實務上會加一個很小的數避免
print("標準化後資料:\n", np.round(standardized, 3))

## 張量 tensor 入門

張量（tensor）就是「可以放在更高維、可被深度學習框架自動運算的多維陣列」。它和 numpy 陣列幾乎是同一回事，只是多了之後會用到的能力（如自動微分的容器）。

用階（rank）描述維度：純量（scalar）是 0 階、向量（vector）是 1 階、矩陣（matrix）是 2 階、再上去就是高維張量。張量同樣有 shape 與 dtype，並能與 numpy 互轉。

本節用 numpy 模擬張量的概念與互轉；若已安裝 PyTorch，原理完全相同。

In [ ]:
# 概念：用 rank（階）區分純量／向量／矩陣，並示範張量與 numpy 的互轉
import numpy as np

scalar = np.array(7.0)                 # 0 階：一個數
vector = np.array([1.0, 2.0, 3.0])     # 1 階：一排數
matrix = np.array([[1.0, 2.0],
                   [3.0, 4.0]])        # 2 階：一張表

# ndim 就是階 rank（有幾個軸）
print("純量 rank:", scalar.ndim, " shape:", scalar.shape)
print("向量 rank:", vector.ndim, " shape:", vector.shape)
print("矩陣 rank:", matrix.ndim, " shape:", matrix.shape)

# 概念：把 numpy 特徵矩陣「當成張量」檢視 shape 與 dtype，再轉回 numpy 確認一致
standardized = (matrix - matrix.mean(axis=0)) / matrix.std(axis=0)
tensor_like = np.asarray(standardized, dtype=np.float32)   # 框架多用 float32，這裡用它模擬張量
print("張量 shape:", tensor_like.shape, " dtype:", tensor_like.dtype)

back_to_numpy = np.asarray(tensor_like)                    # 轉回 numpy
print("轉回 numpy 後資料是否一致:", np.allclose(back_to_numpy, standardized))

## 訓練集與測試集

模型要用「沒看過的資料」來評估才公平。我們把資料切成訓練集（training set）與測試集（test set）：用訓練集學習、用測試集檢驗。

切分前先隨機打亂（shuffle），避免資料原本的排序影響結果；常見比例是八比二。為什麼不能拿測試資料訓練：那等於先看過答案，評估會過度樂觀。

形狀：打亂索引後依比例切片，特徵與標籤要用「同一組索引」對齊。

In [ ]:
# 概念：先打亂再依八比二切成訓練集與測試集，特徵與標籤用同一組索引對齊
import numpy as np

rng = np.random.default_rng(0)   # 固定亂數種子，讓每次執行結果一致（方便對照）

# 造 10 棟建築、各 3 個特徵的假資料
n_samples = 10
features = rng.uniform(low=[5, 50, 0], high=[60, 250, 40], size=(n_samples, 3))
labels = rng.integers(low=0, high=2, size=n_samples)

indices = rng.permutation(n_samples)   # 打亂後的列索引
split = int(n_samples * 0.8)           # 八比二的切點
train_idx, test_idx = indices[:split], indices[split:]

# 注意：特徵與標籤必須用同一組索引切，否則樣本與答案會錯位
X_train, y_train = features[train_idx], labels[train_idx]
X_test, y_test = features[test_idx], labels[test_idx]

print("訓練集樣本數:", X_train.shape[0], " 特徵 shape:", X_train.shape)
print("測試集樣本數:", X_test.shape[0], " 特徵 shape:", X_test.shape)
print("訓練標籤:", y_train, " 測試標籤:", y_test)

## 練習

以下三題由淺入深，皆為複合型但簡短。資料請自己用 numpy 造（仿真即可），完成後對照「驗收」確認輸出。

In [ ]:
# TODO 1 ·（簡單）建築特徵表的形狀檢查（整合：特徵與標籤 + numpy 陣列）
#   情境：你拿到一份自造的社區資料，每棟記錄 樓高 與 面積 兩個特徵，並標記是否為公寓。
#   要求：
#     1. 用 numpy 自造一個 8×2 的特徵矩陣，與一個長度 8 的標籤向量（0/1）。
#     2. 印出特徵矩陣的 shape，並用切片取出所有樓高欄。
#   驗收：應該看到特徵 shape 為 (8, 2)、標籤長度為 8，以及一條只含樓高的一維陣列。
# TODO: 在這裡完成


In [ ]:
# TODO 2 ·（中間）把社區資料標準化並轉成張量（整合：shape 與 dtype + 標準化 + 張量）
#   情境：同一份社區資料中，樓高以公尺、面積以平方公尺記錄，兩個特徵數量級差很大。
#   要求：
#     1. 對特徵矩陣逐欄做 z-score 標準化（用 axis=0 算每欄平均與標準差），確認每欄標準化後平均接近 0。
#     2. 把結果轉成浮點張量（dtype 為 float32），印出張量的 shape 與 dtype，再轉回 numpy 比對是否一致。
#   驗收：應看到標準化後各欄平均接近 0、張量 dtype 為浮點、shape 與原矩陣相同，且轉回 numpy 後數值一致。
# TODO: 在這裡完成


In [ ]:
# TODO 3 ·（稍難）設計一條可重用的資料前處理流程（整合：標準化 + 訓練測試切分 + 形狀／張量）
#   情境：你要為一份較大的自造都市建築資料（含 樓高、面積、容積率 三個特徵）準備可餵入模型的訓練與測試資料。
#   要求：
#     1. 自造 N（例如 20）棟資料，先打亂，再依八比二切成訓練集與測試集。
#     2. 只用「訓練集」算出每欄平均與標準差，並用同一組統計量去標準化訓練集與測試集。
#        （思考：為何不能用測試集自己的統計量？）
#     3. 把兩份標準化後的特徵都轉成浮點張量，印出各自的 shape 與 dtype。
#   驗收：應看到訓練與測試樣本數約為八比二、兩份特徵欄數一致、皆為浮點張量，
#         並能說明用訓練集統計量處理測試集是為了避免資訊洩漏（data leakage）。
# TODO: 在這裡完成


## 小結

- 機器學習的資料通常排成二維表：一列一個樣本、一欄一個特徵，特徵矩陣 shape 為 (樣本數, 特徵數)，標籤為長度 N 的向量。
- numpy 陣列是運算底層；用索引、切片與向量化（搭配廣播）取代迴圈，既快又貼近數學。
- 隨手檢查 shape 與 dtype：reshape 換形狀不改資料，浮點與整數的差別會影響計算精度。
- 特徵標準化用 z-score（減平均、除標準差）讓各特徵尺度一致，避免大數值特徵主導模型。
- 張量與 numpy 陣列幾乎同源，可互轉，並具備之後深度學習要用到的能力。
- 切訓練集與測試集（並先打亂）是公平評估的基本功；切分與依訓練集統計量處理測試集，能避免「背答案」與資訊洩漏。